# 05 · Spatial Neighborhood Analysis

**Dataset:** GSM9936742 · Human Femoral Head · Visium HD (barcodes indicate 56 µm aggregation)  
**Input:** `results/adata/04_clustered.h5ad`  
**Output:** `results/adata/05_spatial.h5ad`, figures and tables.

Analyze **spatial adjacency** independently from the PCA/expression neighbor graph. Compute spatial neighbor enrichment of Leiden domains and Moran’s I for selected genes. These bins are **not individual cells**. QC filtering was not applied in Step 02, so results are exploratory. The tissue image is **not** overlaid because its registration/scale has not been verified.

## 0 · Imports and paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
from scipy import sparse
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "results/adata/04_clustered.h5ad").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run Step 04 first; could not locate results/adata/04_clustered.h5ad")
INPUT = ROOT / "results/adata/04_clustered.h5ad"
OUTPUT = ROOT / "results/adata/05_spatial.h5ad"
FIG = ROOT / "results/figures"
TAB = ROOT / "results/tables"
for p in (FIG, TAB, OUTPUT.parent):
    p.mkdir(parents=True, exist_ok=True)
SEED = 42
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"), "Squidpy:", version("squidpy"))
print("Project root:", ROOT)

## 1 · Load and validate clustered data

In [ ]:
adata = sc.read_h5ad(INPUT)
assert "spatial" in adata.obsm and adata.obsm["spatial"].shape == (adata.n_obs, 2)
assert np.isfinite(adata.obsm["spatial"]).all()
assert "leiden" in adata.obs
assert "connectivities" in adata.obsp and "distances" in adata.obsp, "Expression neighbors missing"
assert "counts" in adata.layers and sparse.issparse(adata.layers["counts"])
assert adata.obs_names.is_unique
adata.obs["leiden"] = adata.obs["leiden"].astype("category")
print("AnnData:", adata.shape)
print("Leiden clusters:", adata.obs["leiden"].nunique())
print("Spatial coordinates range:", np.min(adata.obsm["spatial"], axis=0), np.max(adata.obsm["spatial"], axis=0))

## 2 · Construct a **separate** spatial neighbor graph

Use the physical full-resolution pixel coordinates, **not** `array_row/array_col` as if they were Visium spot-grid coordinates. `coord_type="generic"` with six nearest neighbors avoids incorrectly assuming a Visium hexagonal lattice. Inspect connectivity; nearest-neighbor graphs can link across tissue gaps, so neighborhood results require biological review. The existing expression graph remains in `obsp["connectivities"]`.

In [ ]:
SPATIAL_N_NEIGHBORS = 6
sq.gr.spatial_neighbors(
    adata,
    spatial_key="spatial",
    coord_type="generic",
    n_neighs=SPATIAL_N_NEIGHBORS,
    key_added="spatial",
)
assert "spatial_connectivities" in adata.obsp
assert "spatial_distances" in adata.obsp
assert "connectivities" in adata.obsp
W = adata.obsp["spatial_connectivities"]
assert sparse.issparse(W) and W.shape == (adata.n_obs, adata.n_obs)
assert W.nnz > 0
print("Spatial graph edges (directed stored entries):", W.nnz)
print("Average spatial degree:", float(np.asarray((W > 0).sum(axis=1)).mean()))
print("Expression graph preserved:", adata.obsp["connectivities"].shape)

## 3 · Coordinate-only view of spatial domains

In [ ]:
xy = adata.obsm["spatial"]
labels = adata.obs["leiden"].cat.codes.to_numpy()
ncl = adata.obs["leiden"].nunique()
fig, ax = plt.subplots(figsize=(9, 8))
scat = ax.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="gist_ncar", s=4, alpha=0.8, linewidths=0, rasterized=True, vmin=-0.5, vmax=ncl-0.5)
ax.invert_yaxis()
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Full-resolution pixel X")
ax.set_ylabel("Full-resolution pixel Y (inverted)")
ax.set_title(f"Leiden domains in spatial coordinates ({ncl} clusters)")
fig.colorbar(scat, ax=ax, label="Leiden category index", shrink=0.7)
fig.tight_layout()
fig.savefig(FIG / "05_spatial_domains.png", dpi=180)
plt.show()

## 4 · Spatial neighborhood enrichment

Squidpy permutation-based enrichment compares the observed number of adjacent bins from each pair of clusters to randomized cluster assignments. **Enrichment does not imply cell-cell communication**, especially at 56 µm resolution. A fixed random seed improves reproducibility.

In [ ]:
N_PERMS = 100
sq.gr.nhood_enrichment(
    adata,
    cluster_key="leiden",
    connectivity_key="spatial",
    n_perms=N_PERMS,
    seed=SEED,
    n_jobs=1,
)
result = adata.uns["leiden_nhood_enrichment"]
z = np.asarray(result["zscore"])
cluster_names = adata.obs["leiden"].cat.categories.astype(str).tolist()
assert z.shape == (len(cluster_names), len(cluster_names))
pd.DataFrame(z, index=cluster_names, columns=cluster_names).to_csv(TAB / "05_neighborhood_enrichment_zscores.csv")
fig, ax = plt.subplots(figsize=(max(8, ncl * 0.35), max(7, ncl * 0.32)))
# Cap extreme z-scores only for readable visualization; full values are exported.
heat = ax.imshow(np.clip(z, -10, 10), cmap="coolwarm", vmin=-10, vmax=10, interpolation="nearest")
ax.set_xticks(range(ncl), cluster_names, rotation=90, fontsize=7)
ax.set_yticks(range(ncl), cluster_names, fontsize=7)
ax.set_xlabel("Neighbor cluster")
ax.set_ylabel("Focal cluster")
ax.set_title("Spatial neighborhood enrichment (z-score, display clipped ±10)")
fig.colorbar(heat, ax=ax, shrink=0.75)
fig.tight_layout()
fig.savefig(FIG / "05_neighborhood_enrichment.png", dpi=170)
plt.show()
print("Enrichment z-score matrix:", z.shape)

## 5 · Spatial autocorrelation (Moran’s I) on selected genes

Test a small, transparent panel of human bone/cartilage/marrow-related **candidate** genes where present. This is an exploratory panel, not a validated cell-type annotation. Expression in `adata.X` is normalized log1p expression; no raw-count replacement is performed. Permutations are used for approximate significance. Spatial dependence means these p-values require cautious interpretation.

In [ ]:
CANDIDATE_GENES = ["COL1A1", "COL1A2", "COL2A1", "ACAN", "SPP1", "BGLAP", "RUNX2", "SOX9", "DCN", "LUM", "PTPRC", "PECAM1", "VWF", "MKI67"]
genes = [g for g in CANDIDATE_GENES if g in adata.var_names]
print("Genes available:", genes)
if genes:
    moran = sq.gr.spatial_autocorr(
        adata,
        mode="moran",
        genes=genes,
        connectivity_key="spatial_connectivities",
        n_perms=N_PERMS,
        seed=SEED,
        n_jobs=1,
        copy=True,
    )

    assert moran is not None, "Moran's I calculation returned no result"
    assert not moran.empty, "Moran's I result is empty"

    adata.uns["moranI"] = moran.copy()

    moran.to_csv(TAB / "05_morans_i_candidate_genes.csv")
    print(moran.head(15).to_string())

    fig, ax = plt.subplots(figsize=(9, max(3, len(moran) * 0.38)))
    plot_df = moran.sort_values("I")
    ax.barh(plot_df.index.astype(str), plot_df["I"])
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_xlabel("Moran's I")
    ax.set_title("Spatial autocorrelation of candidate genes")
    fig.tight_layout()
    fig.savefig(FIG / "05_morans_i_candidate_genes.png", dpi=180)
    plt.show()
else:
    print("No candidate genes matched; Moran's I skipped (no fabricated results).")

## 6 · Save and independently reload

In [ ]:
adata.uns["step05_settings"] = {
    "spatial_neighbors": int(SPATIAL_N_NEIGHBORS),
    "coord_type": "generic",
    "nhood_permutations": int(N_PERMS),
    "random_seed": int(SEED),
    "image_registration_verified": False,
    "qc_filters_applied": False,
}
adata.write_h5ad(OUTPUT)
check = sc.read_h5ad(OUTPUT)
assert check.shape == adata.shape
assert check.obs_names.equals(adata.obs_names)
assert check.obs["leiden"].equals(adata.obs["leiden"])
assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
assert "spatial_connectivities" in check.obsp and "spatial_distances" in check.obsp
assert "connectivities" in check.obsp and "distances" in check.obsp
assert "leiden_nhood_enrichment" in check.uns
assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
print("PASS — 05_spatial_analysis completed")
print("AnnData:", check.shape)
print("Spatial graph entries:", check.obsp["spatial_connectivities"].nnz)
print("Leiden clusters:", check.obs["leiden"].nunique())
print("Output:", OUTPUT.relative_to(ROOT))

## Next: `06_marker_analysis.ipynb`

Input: `results/adata/05_spatial.h5ad`. Identify expression-based marker genes by Leiden cluster, while retaining the distinction between spatial **domains** and individual cell types. For statistical comparisons, account for sparse spatial bins, lack of QC filtering, and spatial dependence.